# Platoon RL Full Training (Judge-Ready)

This notebook runs the **full PRD training flow** on Colab GPU: SFT + RL, pushes **LoRA adapters only** to Hugging Face Hub, and generates reward/loss curves.

Expected runtime is long (full run). Use **GPU runtime** and prefer **T4**.

In [ ]:
!nvidia-smi

In [ ]:
# Install project dependencies
!pip install -q -U pip
!pip install -q openenv>=0.1.0 numpy>=1.26.0 torch>=2.3.0 transformers>=4.44.0 trl>=0.11.0 peft>=0.12.0 bitsandbytes>=0.43.0 accelerate>=0.33.0 datasets>=2.20.0 huggingface_hub>=0.24.0 gradio>=4.42.0 matplotlib>=3.9.0 wandb>=0.17.0 tqdm>=4.66.0 pyyaml>=6.0 python-dotenv>=1.0.1

In [ ]:
# Clone repo
REPO_URL = "https://github.com/taruncodes07/SwarmDrive.git"  # TODO: replace
!rm -rf /content/SwarmDrive
!git clone $REPO_URL /content/SwarmDrive
%cd /content/SwarmDrive

## Secrets
Create Colab secrets first: `HF_TOKEN`, `HF_USERNAME` (optional `WANDB_API_KEY`).

Colab: left sidebar -> **Secrets** -> add key/value pairs.

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
hf_username = userdata.get('HF_USERNAME')
wandb_api_key = userdata.get('WANDB_API_KEY') if 'WANDB_API_KEY' in userdata else ''

assert hf_token, 'Missing HF_TOKEN secret'
assert hf_username, 'Missing HF_USERNAME secret'

os.environ['HF_TOKEN'] = hf_token
os.environ['HF_USERNAME'] = hf_username
os.environ['MODEL_ID'] = 'Qwen/Qwen2.5-1.5B-Instruct'
os.environ['LOCAL_FILES_ONLY'] = '0'
os.environ['REQUIRE_CUDA'] = '1'
if wandb_api_key:
    os.environ['WANDB_API_KEY'] = wandb_api_key

with open('.env', 'w', encoding='utf-8') as f:
    f.write(f"HF_USERNAME={hf_username}\n")
    f.write(f"HF_TOKEN={hf_token}\n")
    f.write("MODEL_ID=Qwen/Qwen2.5-1.5B-Instruct\n")
    f.write("LOCAL_FILES_ONLY=0\n")
    f.write("REQUIRE_CUDA=1\n")
    if wandb_api_key:
        f.write(f"WANDB_API_KEY={wandb_api_key}\n")

print('Secrets loaded. .env written for training scripts.')

In [ ]:
# Ensure model repos exist (LoRA-only adapters)
from huggingface_hub import HfApi

api = HfApi(token=os.environ['HF_TOKEN'])
for repo_id in [f"{os.environ['HF_USERNAME']}/platoon-qwen-sft", f"{os.environ['HF_USERNAME']}/platoon-qwen-rl"]:
    api.create_repo(repo_id=repo_id, repo_type='model', private=False, exist_ok=True)
    print('ready:', repo_id)

In [ ]:
# Optional sanity check
!python -m environment.platoon_env --smoke-test

In [ ]:
# Phase 1: SFT (PRD defaults)
# Uses data/sft/scenario_01.jsonl
!python -m training.train_local --sft --epochs 3 --batch-size 2 --grad-accum 8 --lr-sft 2e-4 --report-to auto

In [ ]:
# Phase 2: RL (full training run, PRD defaults)
!python -m training.train_local --rl --episodes 500 --eval-every 50 --checkpoint-every 50 --grpo-update-every 8 --group-size 4 --lr-rl 5e-6 --report-to auto

In [ ]:
from IPython.display import Image, display
from pathlib import Path

print('Artifacts:')
for p in sorted(Path('checkpoints').glob('rl_ep*'))[-5:]:
    print('-', p)

if Path('results/reward_curve.png').exists():
    display(Image(filename='results/reward_curve.png'))
if Path('results/loss_curve.png').exists():
    display(Image(filename='results/loss_curve.png'))